# YOLOv11 Training for RSNA 2024 Lumbar Spine - Sagittal T1 MRI

This notebook trains a YOLOv11x model specifically for **Sagittal T1 MRI images** from the RSNA 2024 Lumbar Spine dataset, following the methodology of the research paper *"YOLOv11 Based Classification of Lumbar Spine Degenerative Changes Across Multi-Modal Imaging"* (Patel et al., 2025).

## Task: Intervertebral Disc Level Detection
- **5 Classes**: L1/L2, L2/L3, L3/L4, L4/L5, L5/S1
- **Image Size**: 384x384 pixels
- **Model**: YOLOv11x (Extra Large)

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q ultralytics pydicom albumentations opencv-python-headless tqdm matplotlib

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import cv2
import pydicom
from tqdm import tqdm
import matplotlib.pyplot as plt
import albumentations as A
from pathlib import Path
import shutil
import yaml
from collections import defaultdict
from ultralytics import YOLO

# Set random seeds for reproducibility
np.random.seed(42)

print("All libraries imported successfully!")

## 2. Data Filtering (Sagittal T1 Focus)

In [ ]:
# Define input path
INPUT_PATH = '/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification'
OUTPUT_PATH = '/kaggle/working/datasets/sagittal_t1'

# Load train series descriptions and filter for Sagittal T1
series_desc_path = f'{INPUT_PATH}/train_series_descriptions.csv'
series_df = pd.read_csv(series_desc_path)

# Filter for Sagittal T1 only
sagittal_t1_df = series_df[series_df['series_description'] == 'Sagittal T1'].copy()
print(f"Total Sagittal T1 series: {len(sagittal_t1_df)}")
print("\nFirst few rows:")
print(sagittal_t1_df.head())

In [ ]:
# Load label coordinates
coords_path = f'{INPUT_PATH}/train_label_coordinates.csv'
coords_df = pd.read_csv(coords_path)

print(f"Total label coordinates: {len(coords_df)}")
print("\nColumns:", coords_df.columns.tolist())
print("\nFirst few rows:")
print(coords_df.head())

# Get unique levels
print("\nUnique levels:", coords_df['level'].unique() if 'level' in coords_df.columns else 'N/A')

## 3. Data Preprocessing & Label Creation

### Class Mapping:
- Class 0: L1/L2
- Class 1: L2/L3
- Class 2: L3/L4
- Class 3: L4/L5
- Class 4: L5/S1

In [ ]:
# Define class mapping for intervertebral disc levels
CLASS_MAPPING = {
    'L1/L2': 0,
    'L2/L3': 1,
    'L3/L4': 2,
    'L4/L5': 3,
    'L5/S1': 4
}

CLASS_NAMES = ['L1/L2', 'L2/L3', 'L3/L4', 'L4/L5', 'L5/S1']

# Fixed bounding box size (in pixels, before scaling)
BOX_SIZE = 32  # 32x32 pixel box around the point

# Target image size
TARGET_SIZE = 384

print(f"Class mapping: {CLASS_MAPPING}")
print(f"Bounding box size: {BOX_SIZE}x{BOX_SIZE} pixels")
print(f"Target image size: {TARGET_SIZE}x{TARGET_SIZE} pixels")

In [ ]:
def normalize_dicom_image(dicom_array):
    """Normalize DICOM pixel values to 8-bit integers (0-255)"""
    # Convert to float for normalization
    image = dicom_array.astype(np.float32)
    
    # Normalize to 0-255 range
    image = image - image.min()
    if image.max() > 0:
        image = image / image.max() * 255.0
    
    return image.astype(np.uint8)


def convert_point_to_yolo_bbox(x, y, img_width, img_height, box_size=32):
    """Convert point coordinates to YOLO format bounding box
    
    Args:
        x, y: Point coordinates in original image
        img_width, img_height: Original image dimensions
        box_size: Size of bounding box in pixels (default: 32)
    
    Returns:
        YOLO format: [x_center, y_center, width, height] (normalized 0-1)
    """
    # Create box centered on point
    half_box = box_size / 2
    
    # Calculate box coordinates
    x_min = max(0, x - half_box)
    x_max = min(img_width, x + half_box)
    y_min = max(0, y - half_box)
    y_max = min(img_height, y + half_box)
    
    # Calculate center and dimensions
    box_width = x_max - x_min
    box_height = y_max - y_min
    x_center = x_min + box_width / 2
    y_center = y_min + box_height / 2
    
    # Normalize to 0-1 range (YOLO format)
    x_center_norm = x_center / img_width
    y_center_norm = y_center / img_height
    width_norm = box_width / img_width
    height_norm = box_height / img_height
    
    return [x_center_norm, y_center_norm, width_norm, height_norm]


def read_and_process_dicom(dicom_path):
    """Read and process a DICOM file"""
    try:
        dcm = pydicom.dcmread(dicom_path)
        image = dcm.pixel_array
        
        # Normalize to 8-bit
        image = normalize_dicom_image(image)
        
        # Convert to RGB (3 channels)
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        return image
    except Exception as e:
        print(f"Error reading {dicom_path}: {e}")
        return None


print("Preprocessing functions defined successfully!")

In [ ]:
def create_yolo_dataset(series_df, coords_df, output_path, train_split=0.8):
    """Create YOLO format dataset from RSNA data
    
    Args:
        series_df: DataFrame with Sagittal T1 series
        coords_df: DataFrame with label coordinates
        output_path: Output directory for dataset
        train_split: Ratio for train/val split (default: 0.8)
    """
    # Create output directories
    train_img_dir = Path(output_path) / 'train' / 'images'
    train_lbl_dir = Path(output_path) / 'train' / 'labels'
    val_img_dir = Path(output_path) / 'val' / 'images'
    val_lbl_dir = Path(output_path) / 'val' / 'labels'
    
    for dir_path in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)
    
    # Group coordinates by study_id and series_id
    coords_grouped = coords_df.groupby(['study_id', 'series_id'])
    
    # Get unique studies for splitting
    unique_studies = series_df['study_id'].unique()
    np.random.shuffle(unique_studies)
    split_idx = int(len(unique_studies) * train_split)
    train_studies = set(unique_studies[:split_idx])
    
    processed_count = 0
    train_count = 0
    val_count = 0
    
    # Process each series
    for idx, row in tqdm(series_df.iterrows(), total=len(series_df), desc="Processing images"):
        study_id = row['study_id']
        series_id = row['series_id']
        
        # Check if this series has coordinates
        if (study_id, series_id) not in coords_grouped.groups:
            continue
        
        # Get coordinates for this series
        series_coords = coords_grouped.get_group((study_id, series_id))
        
        # Build path to DICOM files
        series_path = Path(INPUT_PATH) / 'train_images' / str(study_id) / str(series_id)
        
        if not series_path.exists():
            continue
        
        # Get all DICOM files in the series
        dicom_files = sorted(list(series_path.glob('*.dcm')))
        
        if len(dicom_files) == 0:
            continue
        
        # Group coordinates by instance_number (slice)
        coords_by_instance = defaultdict(list)
        for _, coord_row in series_coords.iterrows():
            instance_num = coord_row['instance_number']
            level = coord_row['level']
            x = coord_row['x']
            y = coord_row['y']
            
            if level in CLASS_MAPPING:
                coords_by_instance[instance_num].append({
                    'level': level,
                    'x': x,
                    'y': y,
                    'class_id': CLASS_MAPPING[level]
                })
        
        # Process each instance with coordinates
        for instance_num, labels in coords_by_instance.items():
            # Find the matching DICOM file
            dicom_file = None
            for dcm_path in dicom_files:
                try:
                    dcm = pydicom.dcmread(dcm_path)
                    if int(dcm.InstanceNumber) == instance_num:
                        dicom_file = dcm_path
                        break
                except:
                    continue
            
            if dicom_file is None:
                continue
            
            # Read and process the image
            image = read_and_process_dicom(dicom_file)
            if image is None:
                continue
            
            orig_height, orig_width = image.shape[:2]
            
            # Resize image to 384x384
            image_resized = cv2.resize(image, (TARGET_SIZE, TARGET_SIZE))
            
            # Calculate scaling factors
            scale_x = TARGET_SIZE / orig_width
            scale_y = TARGET_SIZE / orig_height
            
            # Convert labels to YOLO format
            yolo_labels = []
            for label in labels:
                # Scale coordinates
                x_scaled = label['x'] * scale_x
                y_scaled = label['y'] * scale_y
                
                # Convert to YOLO bbox (on resized image)
                bbox = convert_point_to_yolo_bbox(
                    x_scaled, y_scaled, 
                    TARGET_SIZE, TARGET_SIZE, 
                    box_size=BOX_SIZE
                )
                
                yolo_labels.append(f"{label['class_id']} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}")
            
            if len(yolo_labels) == 0:
                continue
            
            # Determine train/val split
            is_train = study_id in train_studies
            img_dir = train_img_dir if is_train else val_img_dir
            lbl_dir = train_lbl_dir if is_train else val_lbl_dir
            
            # Save image and label
            img_filename = f"{study_id}_{series_id}_{instance_num}.jpg"
            img_path = img_dir / img_filename
            lbl_path = lbl_dir / img_filename.replace('.jpg', '.txt')
            
            # Save image
            cv2.imwrite(str(img_path), cv2.cvtColor(image_resized, cv2.COLOR_RGB2BGR))
            
            # Save label
            with open(lbl_path, 'w') as f:
                f.write('\n'.join(yolo_labels))
            
            processed_count += 1
            if is_train:
                train_count += 1
            else:
                val_count += 1
            
            # Also save to adjacent slices (as per paper's "shared coordinates" strategy)
            # This augments the training data by propagating labels to nearby slices
            for offset in [-1, 1]:
                adj_instance = instance_num + offset
                
                # Find adjacent DICOM file
                adj_dicom_file = None
                for dcm_path in dicom_files:
                    try:
                        dcm = pydicom.dcmread(dcm_path)
                        if int(dcm.InstanceNumber) == adj_instance:
                            adj_dicom_file = dcm_path
                            break
                    except:
                        continue
                
                if adj_dicom_file is None:
                    continue
                
                # Process adjacent image
                adj_image = read_and_process_dicom(adj_dicom_file)
                if adj_image is None:
                    continue
                
                adj_image_resized = cv2.resize(adj_image, (TARGET_SIZE, TARGET_SIZE))
                
                # Save adjacent image with same labels
                adj_img_filename = f"{study_id}_{series_id}_{adj_instance}.jpg"
                adj_img_path = img_dir / adj_img_filename
                adj_lbl_path = lbl_dir / adj_img_filename.replace('.jpg', '.txt')
                
                cv2.imwrite(str(adj_img_path), cv2.cvtColor(adj_image_resized, cv2.COLOR_RGB2BGR))
                
                with open(adj_lbl_path, 'w') as f:
                    f.write('\n'.join(yolo_labels))
                
                processed_count += 1
                if is_train:
                    train_count += 1
                else:
                    val_count += 1
    
    print(f"\nDataset creation complete!")
    print(f"Total processed images: {processed_count}")
    print(f"Training images: {train_count}")
    print(f"Validation images: {val_count}")
    
    return train_count, val_count


# Create the dataset
print("Starting dataset creation...")
train_count, val_count = create_yolo_dataset(sagittal_t1_df, coords_df, OUTPUT_PATH)

## 4. Data Augmentation

Define augmentation pipeline using Albumentations (as per paper: Rotation, Scaling, Flipping)

In [ ]:
# Define augmentation pipeline (will be handled by YOLO during training)
# The paper specifies: Rotation, Scaling, and Flipping

augmentation_pipeline = A.Compose([
    A.Rotate(limit=15, p=0.5),  # Rotation up to ±15 degrees
    A.RandomScale(scale_limit=0.2, p=0.5),  # Scaling ±20%
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.2),  # Occasional vertical flip
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print("Augmentation pipeline defined:")
print("- Rotation: ±15 degrees")
print("- Scaling: ±20%")
print("- Horizontal Flip")
print("- Vertical Flip (occasional)")
print("\nNote: YOLO's built-in augmentation will be used during training.")

## 5. Model Configuration

Create `data.yaml` file for YOLOv11 training

In [ ]:
# Create data.yaml configuration file
data_yaml_content = {
    'path': OUTPUT_PATH,  # Dataset root directory
    'train': 'train/images',  # Train images
    'val': 'val/images',  # Validation images
    'nc': 5,  # Number of classes
    'names': CLASS_NAMES  # Class names
}

data_yaml_path = f'{OUTPUT_PATH}/data.yaml'

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"data.yaml created at: {data_yaml_path}")
print("\nContent:")
print(yaml.dump(data_yaml_content, default_flow_style=False))

In [ ]:
# Verify dataset structure
print("Dataset structure:")
for split in ['train', 'val']:
    img_dir = Path(OUTPUT_PATH) / split / 'images'
    lbl_dir = Path(OUTPUT_PATH) / split / 'labels'
    
    if img_dir.exists():
        num_images = len(list(img_dir.glob('*.jpg')))
        num_labels = len(list(lbl_dir.glob('*.txt')))
        print(f"{split.upper()}: {num_images} images, {num_labels} labels")
    else:
        print(f"{split.upper()}: Directory not found")

## 6. Model Training

Initialize YOLOv11x model and train with exact hyperparameters from the paper (Table III)

In [ ]:
# Initialize YOLOv11x model
model = YOLO('yolo11x.pt')  # Load YOLOv11 Extra Large pretrained model

print("YOLOv11x model loaded successfully!")
print(f"Model: {model.model_name if hasattr(model, 'model_name') else 'YOLOv11x'}")

In [ ]:
# Train the model with hyperparameters from Table III of the paper
results = model.train(
    data=data_yaml_path,
    
    # Training parameters
    imgsz=384,  # Image size: 384x384
    epochs=50,  # Reduced from 100-120 for Kaggle time limits
    batch=16,  # Batch size
    
    # Optimizer settings
    optimizer='AdamW',  # AdamW optimizer
    lr0=0.001,  # Initial learning rate
    weight_decay=0.0005,  # Weight decay
    momentum=0.8,  # Momentum
    
    # Learning rate schedule
    cos_lr=True,  # Cosine annealing learning rate
    warmup_epochs=10,  # Warmup epochs
    
    # Regularization
    dropout=0.2,  # Dropout rate
    
    # Early stopping
    patience=15,  # Early stopping patience
    
    # Data augmentation (built-in YOLO augmentations)
    degrees=15.0,  # Rotation augmentation (±15 degrees)
    scale=0.2,  # Scale augmentation (±20%)
    fliplr=0.5,  # Horizontal flip probability
    flipud=0.2,  # Vertical flip probability
    
    # Other settings
    device=0,  # Use GPU 0
    workers=4,  # Number of dataloader workers
    project='/kaggle/working/runs',  # Output directory
    name='yolov11x_sagittal_t1',  # Run name
    exist_ok=True,  # Allow overwrite
    pretrained=True,  # Use pretrained weights
    verbose=True  # Verbose output
)

print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)

## 7. Evaluation & Visualization

Load the best model and visualize predictions on validation images

In [ ]:
# Load the best trained model
best_model_path = '/kaggle/working/runs/yolov11x_sagittal_t1/weights/best.pt'
best_model = YOLO(best_model_path)

print(f"Best model loaded from: {best_model_path}")

In [ ]:
# Run validation to get metrics
val_results = best_model.val(data=data_yaml_path)

print("\nValidation Results:")
print(f"mAP50: {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall: {val_results.box.mr:.4f}")

In [ ]:
# Visualize predictions on validation images (similar to Figure 9 in the paper)
val_img_dir = Path(OUTPUT_PATH) / 'val' / 'images'
val_images = list(val_img_dir.glob('*.jpg'))[:6]  # Get first 6 validation images

if len(val_images) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(val_images[:6]):
        # Run inference
        results = best_model.predict(source=str(img_path), conf=0.25, save=False, verbose=False)
        
        # Load and display image
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Draw predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy().astype(int)
            confidences = results[0].boxes.conf.cpu().numpy()
            
            for box, cls, conf in zip(boxes, classes, confidences):
                x1, y1, x2, y2 = box.astype(int)
                
                # Draw bounding box
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Draw label
                label = f"{CLASS_NAMES[cls]}: {conf:.2f}"
                (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(image, (x1, y1 - text_height - 4), (x1 + text_width, y1), (0, 255, 0), -1)
                cv2.putText(image, label, (x1, y1 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        
        # Display image
        axes[idx].imshow(image)
        axes[idx].axis('off')
        axes[idx].set_title(f'Sagittal T1 MRI - Image {idx+1}', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/validation_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nVisualization saved to: /kaggle/working/validation_predictions.png")
else:
    print("No validation images found for visualization.")

In [ ]:
# Display training curves
results_csv_path = '/kaggle/working/runs/yolov11x_sagittal_t1/results.csv'

if os.path.exists(results_csv_path):
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = results_df.columns.str.strip()  # Remove whitespace from column names
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot training and validation losses
    if 'train/box_loss' in results_df.columns:
        axes[0, 0].plot(results_df['epoch'], results_df['train/box_loss'], label='Train Box Loss')
        if 'val/box_loss' in results_df.columns:
            axes[0, 0].plot(results_df['epoch'], results_df['val/box_loss'], label='Val Box Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Box Loss')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
    
    # Plot mAP
    if 'metrics/mAP50(B)' in results_df.columns:
        axes[0, 1].plot(results_df['epoch'], results_df['metrics/mAP50(B)'], label='mAP50')
        if 'metrics/mAP50-95(B)' in results_df.columns:
            axes[0, 1].plot(results_df['epoch'], results_df['metrics/mAP50-95(B)'], label='mAP50-95')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('mAP')
        axes[0, 1].set_title('Mean Average Precision')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    # Plot precision and recall
    if 'metrics/precision(B)' in results_df.columns:
        axes[1, 0].plot(results_df['epoch'], results_df['metrics/precision(B)'], label='Precision')
        if 'metrics/recall(B)' in results_df.columns:
            axes[1, 0].plot(results_df['epoch'], results_df['metrics/recall(B)'], label='Recall')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Score')
        axes[1, 0].set_title('Precision and Recall')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    
    # Plot class loss
    if 'train/cls_loss' in results_df.columns:
        axes[1, 1].plot(results_df['epoch'], results_df['train/cls_loss'], label='Train Class Loss')
        if 'val/cls_loss' in results_df.columns:
            axes[1, 1].plot(results_df['epoch'], results_df['val/cls_loss'], label='Val Class Loss')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Class Loss')
        axes[1, 1].set_title('Classification Loss')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nTraining curves saved to: /kaggle/working/training_curves.png")
else:
    print(f"Results CSV not found at: {results_csv_path}")

## Summary

This notebook has successfully:

1. **Filtered** the RSNA 2024 dataset to only Sagittal T1 MRI images
2. **Preprocessed** DICOM images (normalization, resizing to 384x384)
3. **Created** YOLO-format bounding box labels from point coordinates
4. **Mapped** 5 intervertebral disc level classes (L1/L2, L2/L3, L3/L4, L4/L5, L5/S1)
5. **Applied** shared coordinates strategy (label propagation to adjacent slices)
6. **Trained** YOLOv11x model with exact hyperparameters from the paper
7. **Evaluated** and **visualized** results similar to Figure 9 of the paper

### Key Hyperparameters (from Table III):
- Image size: 384×384
- Batch size: 16
- Optimizer: AdamW
- Learning rate: 0.001 (with cosine annealing)
- Weight decay: 0.0005
- Dropout: 0.2
- Momentum: 0.8
- Warmup epochs: 10
- Patience: 15

### Output Files:
- Best model: `/kaggle/working/runs/yolov11x_sagittal_t1/weights/best.pt`
- Training results: `/kaggle/working/runs/yolov11x_sagittal_t1/results.csv`
- Validation predictions: `/kaggle/working/validation_predictions.png`
- Training curves: `/kaggle/working/training_curves.png`